In [1]:
import torch
from load_data import domain_var, load_data, temperature_var, velocity_var

import qewton

Define Hyperparameters

In [2]:
batch_size = qewton.optim.DiscreteHyperparameter((10, 50))
pca_n_in = qewton.optim.DiscreteHyperparameter((10, 100), default_grid=10)
pca_n_out = qewton.optim.DiscreteHyperparameter((10, 100), default_grid=10)
n_layers = qewton.optim.DiscreteHyperparameter((1, 4), default_grid=1)
n_neurons = qewton.optim.DiscreteHyperparameter((10, 100), default_grid=10)
lr = qewton.optim.ContinuousHyperparameter(
    (1e-4, 1e-2), scale=qewton.optim.HyperParameterScale.LOG
)


Load data

In [3]:
dataset, data_loader = load_data(batch_size)
print(domain_var)
print(temperature_var)
print(velocity_var)


--- Data quality report ---
Total samples:     581
Valid samples:     580
Discarded samples: 1

Discarded samples:
  Sample 40: pressure: max abs = 8.7486e+14 (limit 1000000.0)
Variable(domain, dim=1)
Variable(temperature, dim=1)
Variable(velocity, dim=3, children=3)


Setup computation graph

In [4]:
graph = qewton.Graph()
model = qewton.algorithms.PCANet(
    input_variable=domain_var,
    output_variable=temperature_var * velocity_var,
    pca_n_input=pca_n_in,
    pca_n_output=pca_n_out,
    data_source_node=data_loader,
    fcn_hidden_layers=n_layers,
    fcn_hidden_neurons=n_neurons,
)
concat = qewton.algorithms.building_blocks.ConcatVariables(
    [temperature_var, velocity_var],
    name="Concat temperature and velocity data",
)
split = qewton.algorithms.building_blocks.SplitVariables(
    [temperature_var, velocity_var],
    name="Split temperature and velocity prediction",
)
temperature_constraint = qewton.constraints.MSEConstraint(name="Temperature MSE")
velocity_constraint = qewton.constraints.MSEConstraint(name="Velocity MSE", weight=1/3) # higher dimension than temperature

# input features
graph.connect(
    data_loader.get_output_port(domain_var), model.input_ports[0]
)
# output features
graph.connect(
    data_loader.get_output_port(temperature_var),
    concat.get_input_port(temperature_var),
)
graph.connect(
    data_loader.get_output_port(velocity_var),
    concat.get_input_port(velocity_var),
)
graph.connect(
    concat.output_ports[0], model.input_ports[1]
)
# constraints
graph.connect(
    model, split.input_ports[0]
)
graph.connect(
    split.get_output_port(temperature_var), temperature_constraint.input_1
)
graph.connect(
    data_loader.get_output_port(temperature_var), temperature_constraint.input_2
)
graph.connect(
    split.get_output_port(velocity_var), velocity_constraint.input_1
)
graph.connect(
    data_loader.get_output_port(velocity_var), velocity_constraint.input_2
)


Setup trainer

In [5]:
devices = [f"cuda:{i}" for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else ["cpu"]
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam(),
    lr=lr,
    max_iterations=2500,
)
trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=[adam_phase],
    graphs=[graph],
    training_objectives=[temperature_constraint, velocity_constraint],
    callbacks=[qewton.optim.TensorboardLogger()],
    device=devices[0]
)


Start with grid search tuning

In [6]:
tuner = qewton.optim.tuner.GridSearchTuner(
    trainer,
    tuning_objectives=trainer.training_objectives,
    trial_number=50,
    devices=devices,
    trials_per_device=2,
    save_path="tuner_results/pcanet_random",
    tuning_callbacks=[
        qewton.optim.tuner.EarlyStoppingTuneCallback(monitor_constraint=temperature_constraint),
        qewton.optim.tuner.EarlyStoppingTuneCallback(monitor_constraint=velocity_constraint),
    ],
)
tuner.run()

--- Start Tuning ---
Working on trials 0 - 10
Working on trials 10 - 20
Working on trials 20 - 30
Working on trials 30 - 40
Working on trials 40 - 50
Working on trials 50 - 50
--- Cleaning up ---


[W1119 15:34:06.749942708 CudaIPCTypes.cpp:16] Producer process has been terminated before all shared CUDA tensors released. See Note [Sharing CUDA tensors]
[W1119 15:34:06.757967195 CudaIPCTypes.cpp:16] Producer process has been terminated before all shared CUDA tensors released. See Note [Sharing CUDA tensors]
[W1119 15:34:06.758469485 CudaIPCTypes.cpp:16] Producer process has been terminated before all shared CUDA tensors released. See Note [Sharing CUDA tensors]
[W1119 15:34:06.767830232 CudaIPCTypes.cpp:16] Producer process has been terminated before all shared CUDA tensors released. See Note [Sharing CUDA tensors]
[W1119 15:34:06.771973561 CudaIPCTypes.cpp:16] Producer process has been terminated before all shared CUDA tensors released. See Note [Sharing CUDA tensors]
[W1119 15:34:06.789647575 CudaIPCTypes.cpp:16] Producer process has been terminated before all shared CUDA tensors released. See Note [Sharing CUDA tensors]
[W1119 15:34:06.807693319 CudaIPCTypes.cpp:16] Producer pr

--- Finished Tuning ---
